# Tensor Algebra and Coordinate Transformation

This notebook demonstrates key tensor algebra operations in Python, focusing on how tensor components change under basis transformation and how contraction reduces tensor order.

We work with explicit tensor components in low-dimensional vector spaces and show how these abstract operations can be implemented and verified numerically.

In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

## Tensor coordinate transformation in $\mathbb{R}^3$

For $V=\mathbb{R}^3$, define a rank-4 tensor by its components in a chosen basis:
$$
T_{ijkl} = ((i+1) + (j+1)b)(k+1) + (l+1),
$$
with a fixed scalar $b$. This component formula is independent of the ordering of covariant and contravariant indices for the coordinate manipulations shown here.

In [2]:
scalar_b = 0.3667
change_matrix = np.array([[0.452, -1.0099, 0.872],
                          [-1.0099, 0.0764, 0.7095],
                          [0.872, 0.7095, 0.4715]])
change_matrix_inv = np.linalg.inv(change_matrix)

T = np.zeros((3, 3, 3, 3))
for i in range(3):
    for j in range(3):
        for k in range(3):
            for l in range(3):
                T[i, j, k, l] = ((i + 1) + (j + 1) * scalar_b) * (k + 1) + (l + 1)

Notice that this representation does not specify which indices are covariant and which are contravariant. 

For a tensor of type $(2,2)$, the change of basis uses the coordinate transformation matrix and its inverse to compute the new components.

In [3]:
T22 = np.zeros((3, 3, 3, 3))
for w in range(3):
    for x in range(3):
        for y in range(3):
            for z in range(3):
                for i in range(3):
                    for j in range(3):
                        for k in range(3):
                            for l in range(3):
                                T22[w, x, y, z] += T[i, j, k, l] * change_matrix[w, i] * change_matrix[x, j] * change_matrix_inv[y, k] * change_matrix_inv[z, l]
print(T22[1, 0, 1, 0])

0.036723516101578646


For a tensor of type $(4,0)$, all four indices transform with the change-of-basis matrix.

In [4]:
T40 = np.zeros((3, 3, 3, 3))
for w in range(3):
    for x in range(3):
        for y in range(3):
            for z in range(3):
                for i in range(3):
                    for j in range(3):
                        for k in range(3):
                            for l in range(3):
                                T40[w, x, y, z] += T[i, j, k, l] * change_matrix_inv[w, i] * change_matrix_inv[x, j] * change_matrix_inv[y, k] * change_matrix_inv[z, l]
print(T40[1, 2, 1, 2])

8.633467696689213


For a tensor of type $(0,4)$, all four indices transform with the inverse change-of-basis matrix.

In [5]:
T04 = np.zeros((3, 3, 3, 3))
for w in range(3):
    for x in range(3):
        for y in range(3):
            for z in range(3):
                for i in range(3):
                    for j in range(3):
                        for k in range(3):
                            for l in range(3):
                                T04[w, x, y, z] += T[i, j, k, l] * change_matrix[w, i] * change_matrix[x, j] * change_matrix[y, k] * change_matrix[z, l]
print(T04[0, 2, 2, 0])

4.389089707962575


## Tensor contraction and index reduction

Contraction reduces the order of a tensor by pairing a covariant index with a contravariant index. For $T\in \mathcal{T}^{r+1}_{s+1}$ and a given pair of positions $(p,q)$, the contracted tensor $C_q^p T \in \mathcal{T}^r_s$ is obtained by summing over one repeated index.

In coordinates, this operation removes one covariant and one contravariant index from the component array.

Consider a $(2,2)$ tensor $P$ on $V=\mathbb{R}^2$. It can also be viewed as a $(1,1)$ tensor on the tensor-product space $V\otimes V=\mathbb{R}^4$, so its coordinate representation corresponds to a $4\times4$ matrix.

If we assume that we begin with a $(2,2)$--tensor $\rho$, the result of a $(1,1)$--contraction,  $\rho_1=C_1^1 \rho$, reads

In [6]:
P = np.array([[0.3639, 0.6942, 0.681, 0.317],
              [0.0725, 0.7793, 0.1233, 0.9223],
              [0.6691, 0.4955, 0.0621, 0.8234],
              [0.3771, 0.4876, 0.1819, 0.1024]])

# Compute the (1,1) contraction of the (2,2) tensor represented by the 4x4 matrix P
P1 = np.zeros((2, 2))
P1[0, 0] = P[0, 0] + P[2, 2]
P1[0, 1] = P[0, 1] + P[2, 3]
P1[1, 0] = P[1, 0] + P[3, 2]
P1[1, 1] = P[1, 1] + P[3, 3]

print(P1[1, 0])

0.2544


The next code block computes one component of the contracted tensor coordinates.

Analogously, the (2,2)-contraction defines a (1,1) tensor on $V$, $\rho_2=C_2^2 \rho$.

In [7]:
# Alternative contraction pattern for the original 4x4 tensor
P2 = np.zeros((2, 2))
P2[0, 0] = P[0, 0] + P[1, 1]
P2[0, 1] = P[0, 2] + P[1, 3]
P2[1, 0] = P[2, 0] + P[3, 1]
P2[1, 1] = P[2, 2] + P[3, 3]

Under a change of basis on $\mathbb{R}^2$ defined by a matrix $C_2$, the contracted tensor coordinates transform accordingly. The following code computes the transformed component of the resulting (1,1) tensor.

In [8]:
C2 = np.array([[0.1, 0.6789], [0.3278, 0.2731]])
invC2 = np.linalg.inv(C2)
P2C = np.zeros((2, 2))
for x in range(2):
    for y in range(2):
        for i in range(2):
            for j in range(2):
                P2C[x, y] += P2[i, j] * C2.T[x, i] * invC2[y, j]

print(P2C[1, 1])

1.2529689402050124


The computed component is printed below for verification.

In [9]:
# Equivalent matrix expression for the same transformed contracted tensor
P2C = C2.T @ P2 @ invC2.T
print(P2C[1, 1])

1.2529689402050121


### Summary

This notebook demonstrates:
- tensor coordinate transformation under basis change,
- explicit component transformation for types $(2,2)$, $(4,0)$, and $(0,4)$,
- tensor contraction as index reduction,
- how contracted tensors transform under a secondary basis change.

The code is designed for clarity, reproducibility, and verification of tensor algebra concepts.